In [2]:
%matplotlib inline

In [3]:
# Imports
import os
import pathlib
SALVUS_FLOW_SITE_NAME = os.environ.get("macbook") # Site name given in the installation of Salvus flow
PROJECT_DIR = "simulation_wavefield_moving_source"  

# Add code to keep .gitignore updated to ignore salvus files
gitignore_path = pathlib.Path("..") / ".gitignore"
with open(gitignore_path, "r+") as f:
    contents = f.read()
    if PROJECT_DIR not in contents:
        f.write(f"\n{PROJECT_DIR}/\n")


import numpy as np
import salvus.namespace as sn
import xarray as xr
import salvus.namespace as sn
from salvus.project.tools.processing import block_processing
from salvus.toolbox.helpers.wavefield_output import (
    WavefieldOutput,
    wavefield_output_to_xarray,
)
import matplotlib.pyplot as plt
from matplotlib import animation
import obspy

# For wavefield output code
from salvus.mesh.unstructured_mesh_utils import read_model_from_h5
from salvus.toolbox.helpers import wavefield_output

#for plotting of wiggles, traces 
from scipy import signal

# For animation plot
from IPython.display import HTML
from matplotlib.collections import PolyCollection
import threading
import matplotlib
matplotlib.use("Agg")
from scipy.interpolate import griddata

--> Server: 'https://l.mondaic.com/licensing_server', User: 'salome.bachmann', Group: 'ETHZ_ERDW_EEG'.
--> Negotiating 1 license instance(s) for 'SalvusMesh' [license version 1.0.0] for 1 seconds ...
--> Success! [Total duration: 0.84 seconds]


In [4]:
# Setup of the model domain as a box (same as research module)
domain_2d = sn.domain.dim2.BoxDomain(x0=0, x1=300, y0=0, y1=3)
p = sn.Project.from_domain(path=PROJECT_DIR, domain=domain_2d, load_if_exists=True)

Accordion()

In [5]:

# Layered model setup according to mondaic docs
# Minimal and maximal x extent: same as domain box
x_min = 0.0
x_max = 300.0

# Defining extent of löayers (layers_x) and thickness / topography of layers (layers_y)
layers_x = [
    np.array([0.0, 300.0]),  # top boundary
    np.array([0.0, 300.0]),  # snow-air interface
    np.array([0.0, 300.0]),  # earth-snow interface 
    np.array([0.0, 300.0]),  # bottom boundary
]

layers_y = [
    np.array([3.0, 3.0]),        
    np.array([3*2/3, 3*2/3]),        
    np.array([3*1/3, 3*1/3]),        
    np.array([0.0, 0.0]),        
]

# Defining model parameters (vp, vs and rho) for earth, snow and air, earth and air velocities taken from https://pburnley.faculty.unlv.edu/GEOL452_652/seismology/notes/SeismicNotes10RVel.html
vp = np.array([2200, 300, 332])
#vs = np.array([0,0,0])
vs = np.array([880, 150,0])
rho = np.array([2000, 180, 1.2250])


interpolation_styles = ["linear"] * len(layers_x)


splines = sn.toolbox.get_interpolating_splines(
    layers_x, layers_y, kind=interpolation_styles
)

# # Plotting the layer boundaries to check if they are correct
# f = plt.figure(figsize=(10, 5))
# x_plot = np.linspace(x_min, x_max)
# for top, bot in splines:
#     plt.plot(x_plot, top(x_plot))
#     plt.plot(x_plot, bot(x_plot))

# plt.xlabel("x (m)")
# plt.ylabel("y (m)")
# plt.title("Interfaces")
# plt.ylim(0,1.5)

# Genetarte mesh
# Maximum frequency to resolve with elements_per_wavelength.
max_frequency = 50.0

# Print lenght of splines because of size mismatch between splines and vs
shp = len(splines)
print(shp)

slowest_velocities = np.array([
    880,   # earth
    150,   # snow
    150,   # air layer meshing controlled by snow below --> need this because else slowest_velocities gives an errror because it goes to infinity
])

# Generate the mesh
mesh, bnd = sn.toolbox.generate_mesh_from_splines_2d(
    x_min=0,
    x_max=x_max,
    splines=splines,
    elements_per_wavelength=2,
    maximum_frequency=max_frequency,
    use_refinements=True,
    slowest_velocities=slowest_velocities,
    # make very bottom boundary, very top (in x) and both sides in y absorbing
    absorbing_boundaries=(["x0", "x1", "y0", "y1"], 10.0), # Change this if different boundaries need to be absorbing CHECK 
)

mesh = np.sum(mesh)

# Add info about absorbing boundaries CHANGE DEPENDING ON WHICH BOUNDARIES NEED TO BE TRANSPARENT / ABSORBING
mesh.attach_global_variable("max_dist_ABC", bnd)
mesh.attach_global_variable("ABC_side_sets", ", ".join(["x0", "x1", "y0"]))
mesh.attach_global_variable("ABC_vel", float(min(vs[vs > 0])))
mesh.attach_global_variable("ABC_freq", max_frequency / 2.0)
mesh.attach_global_variable("ABC_nwave", 5.0)


# Attaching parameters (vp,vs,rho) to mesh 
nodes = mesh.get_element_nodes()[:, :, 0]
vp_a, vs_a, ro_a = np.ones((3, *nodes.shape))
for _i, (vp_val, vs_val, ro_val) in enumerate(zip(vp, vs, rho)):
    # Find which elements are in a given region.
    idx = np.where(mesh.elemental_fields["region"] == _i)

    # Set parameters in that region to a constant value.
    vp_a[idx] = vp_val
    vs_a[idx] = vs_val
    ro_a[idx] = ro_val

# Attach parameters.
for k, v in zip(["VP", "VS", "RHO"], [vp_a, vs_a, ro_a]):
    mesh.attach_field(k, v)

# Attach acoustic / elastic flag.
mesh_2d_layered = sn.toolbox.detect_fluid(mesh)

# # Checking which values are assigned to which layer: LAYER 0 IS THE BOTTOM LAYER
# np.unique(mesh.elemental_fields["region"])
# for i in range(3):
#     idx = mesh.elemental_fields["region"] == i
#     print(i,
#           np.unique(mesh.elemental_fields["VP"][idx]),
#           np.unique(mesh.elemental_fields["VS"][idx]),
#           np.unique(mesh.elemental_fields["RHO"][idx]))


# # Plot Mesh toc heck
#mesh_2d_layered



3


In [6]:
# Moving source setup: create one event per x-position along the domain
x_positions = np.arange(0.0, 300.0, 15.0)  # Start coordinate, End coordinate, step size
y_source = 1.5

moving_source_event_names = []
for i, x_src in enumerate(x_positions):
    src = sn.simple_config.source.cartesian.VectorPoint2D(
        x=float(x_src),
        y=y_source,
        fx=0.0, # same forces as previously 
        fy=-1.0,
    )

    event_name = f"event_wavefield_output_x{i:03d}"
    p.add_to_project(sn.Event(event_name=event_name, sources=[src]))
    moving_source_event_names.append(event_name)

print(f"Added {len(moving_source_event_names)} moving-source events.")
print(f"First event: {moving_source_event_names[0]}")
print(f"Last event:  {moving_source_event_names[-1]}")

Added 20 moving-source events.
First event: event_wavefield_output_x000
Last event:  event_wavefield_output_x019


In [7]:
p.add_to_project(
    sn.UnstructuredMeshSimulationConfiguration(
        name="sim_2d_layered_moving_source",
        unstructured_mesh=mesh_2d_layered,
        event_configuration=sn.EventConfiguration(
            wavelet=sn.simple_config.stf.Ricker(
                center_frequency=10,
                time_shift_in_seconds=0.3  # shifts wavelet 
            ),
            waveform_simulation_configuration=sn.WaveformSimulationConfiguration(
                start_time_in_seconds=-0.3,
                end_time_in_seconds=2.0,
            ),
        ),
    ),
)

In [9]:
# Layered
p.simulations.launch(
    simulation_configuration="sim_2d_layered_moving_source",
    events=p.events.list(),
    site_name="macbook", 
    ranks_per_job=4,
    extra_output_configuration={
        "volume_data": {
            "sampling_interval_in_time_steps": 50,
            "fields": ["velocity", "displacement"], # add displacement to field 
        },
    },
)
p.simulations.query(block=True)

[2026-04-13 10:32:35,594] INFO: Submitting job array with 20 jobs ...


Initializing Jobs and Uploading Files for Individual Jobs:   0%|          | 0/20 [00:00<?, ?job/s]

VBox()

True

In [10]:
# Extracting wavefield output for all moving source events
vel_outputs = []

for event_name in moving_source_event_names:
    out_dir = p.simulations.get_simulation_output_directory("sim_2d_layered_moving_source", event_name)
    
    vel_wo = wavefield_output.WavefieldOutput.from_file(
        pathlib.Path(out_dir, "volume_data_output.h5"),
        "velocity",
        "volume",
    )
    
    vel_xr = wavefield_output.wavefield_output_to_xarray(
        vel_wo,
        points=[np.linspace(0, 300, 301), np.linspace(0, 3, 101)],
    )
    
    vel_outputs.append(vel_xr)

# Combine all events into a single xarray along new event dimension
vel_2d_layered = xr.concat(vel_outputs, dim="event_index")
vel_2d_layered = vel_2d_layered.assign_coords(event_index=("event_index", x_positions))

print(f"Combined {len(vel_outputs)} moving-source events into single xarray")
print(f"Shape: {vel_2d_layered.dims}")
print(vel_2d_layered)

Finding enclosing elements (pass 1 of auto):   0%|          | 0/30401 [00:00<?, ?it/s]

Finding enclosing elements (pass 2 of auto):   0%|          | 0/20468 [00:00<?, ?it/s]

Finding enclosing elements (pass 3 of auto):   0%|          | 0/20468 [00:00<?, ?it/s]

Finding enclosing elements (pass 4 of auto):   0%|          | 0/20468 [00:00<?, ?it/s]

Finding enclosing elements (pass 5 of auto):   0%|          | 0/10234 [00:00<?, ?it/s]

...

Finding enclosing elements (pass 8 of auto):   0%|          | 0/10234 [00:00<?, ?it/s]

Finding enclosing elements (pass 9 of auto):   0%|          | 0/10234 [00:00<?, ?it/s]

[2026-04-13 10:53:26,671] WARNING - salvus.mesh.algorithms.unstructured_mesh.utils: 10234 points were not claimed by enclosing elements. Depending on your use case, this may not be an issue.


Extracting wavefield to regular grid:   0%|          | 0/11 [00:00<?, ?it/s]

Combined 20 moving-source events into single xarray
Shape: ('event_index', 't', 'c', 'x', 'y')
<xarray.DataArray (event_index: 20, t: 470, c: 2, x: 301, y: 101)> Size: 2GB
array([[[[[            nan,             nan,             nan, ...,
            0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
          [            nan,             nan,             nan, ...,
            0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
          [            nan,             nan,             nan, ...,
            0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
          ...,
          [            nan,             nan,             nan, ...,
            0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
          [            nan,             nan,             nan, ...,
            0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
          [            nan,             nan,             nan, ...,
            0.00000000e+00,  0.00000000e+00,  0.00000000e+00]],

         [[            nan,    

In [ ]:

# Define receiver line at snow-earth interface
y_surface = 3 * 2 / 3  # middle of snow layer 
#y_surface = 3 * 1 / 3 # snow-air interface
x_line = np.linspace(0.0, 300.0, 1001)
y_line = np.full_like(x_line, y_surface)

# Extract wavefield along receiver line
vel_sg = wavefield_output.wavefield_output_to_xarray(
    vel_wo_layered,
    points=np.column_stack((x_line, y_line)),
)

# Select vertical component and full time range
sg_vy = vel_sg.isel(c=1)

t_vals = sg_vy.t.values
data = sg_vy.values  # shape: (n_t, n_points)

print("t range:", t_vals[0], "->", t_vals[-1])
print("data shape:", data.shape)

# Clip colorscale (robust equivalent)
vmax = np.percentile(np.abs(data), 95)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.pcolormesh(
    x_line, t_vals, data,
    shading="gouraud",
    cmap="RdBu_r",
    vmin=-vmax, vmax=vmax,
)
ax.invert_yaxis()
ax.axvline(x=150, color="teal", lw=0.8, linestyle="--", label="source x=15")
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Time (s)")
ax.set_xlim(0, 300)
ax.set_title("Seismic waterfall plot - vertical particle velocity at halfway depth in snow (middle of snow layer)")
#ax.set_title("Seismic waterfall plot - vertical particle velocity at snow-earth interface")
ax.legend()
plt.colorbar(im, ax=ax, label="Vertical particle velocity (m/s)")
plt.tight_layout()
plt.show()

In [ ]:

# extract raw data
crds     = vel_2d_layered.crds # (n_elements, n_nodes, 2)
data_raw = vel_2d_layered.data # (n_time, n_elements, n_components, n_nodes)
t_sr     = vel_2d_layered.sampling_rate
t_start  = vel_2d_layered.start_time

# flatten coordinates to (N_points, 2)
n_elem, n_node, _ = crds.shape
xy_flat = crds.reshape(-1, 2)
x_pts   = xy_flat[:, 0]
y_pts   = xy_flat[:, 1]

# number of time steps and time axis
n_t    = data_raw.shape[0]
t_vals = t_start + np.arange(n_t) / t_sr

# vy component flattened to (n_t, N_points)
vy_flat = data_raw[:, :, 1, :].reshape(n_t, -1)

# coordinate setup: salvus coordinates do not macth initially defined model coordinates
y_min_s = y_pts.min() # 1.0  in Salvus coords
y_max_s = y_pts.max() # 99.8 in Salvus coords
x_min_s = x_pts.min()  # -16.5
x_max_s = x_pts.max()  # 316.5
scale   = (y_max_s - y_min_s) / 3.0# 32.93 - Salvus units per model metre

# interface and source positions in Salvus coordinates
y_interface_snow_firn = y_min_s + (3*1/3) * scale # = 33.93
y_interface_firn_bed  = y_min_s + (3*2/3) * scale # = 66.87
y_source_s            = y_min_s + 1.5     * scale # = 17.47 (mid snow layer)
x_source_s            = 150.0

# grid interpolation 
nx, ny = 300, 80
x_grid = np.linspace(x_min_s, x_max_s, nx)
y_grid = np.linspace(y_min_s, y_max_s, ny)
xx, yy = np.meshgrid(x_grid, y_grid)

# time subsamplin 
t_start_idx = np.searchsorted(t_vals, 0.0)
N           = 2  # increase to speed up, decrease for smoother animation
t_idx       = np.arange(t_start_idx, n_t, N)

print(f"x range:        {x_min_s:.1f} → {x_max_s:.1f} m")
print(f"y range:        {y_min_s:.1f} → {y_max_s:.1f} m")
print(f"scale:          {scale:.2f} Salvus units per model metre")
print(f"t range:        {t_vals[0]:.3f} → {t_vals[-1]:.3f} s")
print(f"t_start_idx:    {t_start_idx}  (t={t_vals[t_start_idx]:.3f} s)")
print(f"frames to anim: {len(t_idx)}")
print(f"snow-soil:      y = {y_interface_snow_firn:.2f} (model 1.0 m)")
print(f"bottom of soil:   y = {y_interface_firn_bed:.2f}  (model 2.0 m)")
print(f"source:         x = {x_source_s:.1f}, y = {y_source_s:.2f} (model 1.5 m)")


# backgorund thread function for animation
def run_animation():

    # pre-interpolate all frames 
    print("Pre-interpolating frames...", flush=True)
    frames_2d = []
    for k, i in enumerate(t_idx):
        grid_vals = griddata(
            xy_flat, vy_flat[i],
            (xx, yy),
            method="linear",
            fill_value=0.0,
        )
        frames_2d.append(grid_vals.astype(np.float32))
        if k % 20 == 0:
            print(f"  frame {k}/{len(t_idx)}  t={t_vals[i]:.3f} s", flush=True)

    frames_2d = np.array(frames_2d)   # (n_frames, ny, nx)

    # clip colorscale using later frames so source doesn't dominate
    vmax = np.percentile(np.abs(frames_2d[len(frames_2d)//4:]), 98)
    print(f"Interpolation done. vmax = {vmax:.2e}", flush=True)

    # figure setu
    fig, ax = plt.subplots(figsize=(14, 5))

    im = ax.imshow(
        frames_2d[0],
        extent=[x_grid[0], x_grid[-1], y_grid[-1], y_grid[0]],
        aspect="auto",
        cmap="RdBu_r",
        vmin=-vmax, vmax=vmax,
        origin="upper",
        interpolation="bilinear",
    )

    # interface lines
    ax.axhline(y_interface_snow_firn, color="black", lw=1.2,
               linestyle="--", label="air-snow (1.0 m)")
    ax.axhline(y_interface_firn_bed,  color="gray",  lw=1.2,
               linestyle="--", label="snow-soil (2.0 m)")

    # source marker
    ax.plot(x_source_s, y_source_s, "v", color="teal",
            ms=8, label="source (x=150, y=1.5 m)")

    # y axis ticks in real model coordinates
    model_depths   = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
    salvus_y_ticks = [y_min_s + d * scale for d in model_depths]
    ax.set_yticks(salvus_y_ticks)
    ax.set_yticklabels([f"{d} m" for d in model_depths])

    # layer name annotations
    ax.text(x_min_s + 5, y_min_s + 0.10 * scale, "",
            fontsize=8, color="gray")
    ax.text(x_min_s + 5, y_min_s + 0.55 * scale, "air",
            fontsize=8, color="steelblue")
    ax.text(x_min_s + 5, y_min_s + 1.55 * scale, "snow",
            fontsize=8, color="steelblue")
    ax.text(x_min_s + 5, y_min_s + 2.55 * scale, "",
            fontsize=8, color="steelblue")

    ax.set_xlabel("x (m)")
    ax.set_ylabel("Depth (m)")
    ax.set_ylim(y_grid[-1], y_grid[0])
    ax.legend(loc="upper right", fontsize=8)
    plt.colorbar(im, ax=ax,
                 label="Vertical particle velocity vy (m/s)",
                 shrink=0.8)
    title = ax.set_title(f"Wavefield- t = {t_vals[t_start_idx]:.4f} s")
    plt.tight_layout()

    # animation update function
    def update(frame_idx):
        im.set_data(frames_2d[frame_idx])
        title.set_text(f"Wavefield - t = {t_vals[t_idx[frame_idx]]:.4f} s")
        return im, title

    ani = animation.FuncAnimation(
        fig, update,
        frames=len(t_idx),
        interval=40,
        blit=True,
    )

    # save animation functon 
    print("Saving animation...", flush=True)
    ani.save(
        "wavefield_2d.mp4",
        writer="ffmpeg",
        fps=30,
        dpi=120,
        extra_args=["-vcodec", "libx264", "-pix_fmt", "yuv420p"],
        progress_callback=lambda i, n: print(
            f"  saving frame {i}/{n}", flush=True
        ) if i % 50 == 0 else None,
    )
    plt.close(fig) # to close initail frame 
    print("Done! Saved as wavefield_2d.mp4", flush=True)


#launch background thread 
thread = threading.Thread(target=run_animation)
thread.start()
print("Animation running in background thread.")
print("Check progress with: thread.is_alive()")